# Calling CUDA Functions from Python — 3 Ways
**Week 9 Activity** | Run on Google Colab with GPU Runtime enabled

This notebook demonstrates three methods for calling a CUDA C++ function from Python:

| # | Method | Description |
|---|--------|-------------|
| 1 | **ctypes** | Python built-in module loads the compiled `.so` directly — no build step needed |
| 2 | **C Extension Wrapper** | Custom C module wraps the CUDA library; accepts Python lists |
| 3 | **NumPy C Extension Wrapper** | Same idea but passes NumPy arrays directly — more efficient |

The CUDA kernel performs **element-wise multiplication** of two integer arrays on the GPU.

> **Before running:** Go to `Runtime > Change runtime type` and select **T4 GPU**.

In [ ]:
# Step 1: Verify GPU is available
!nvidia-smi

In [ ]:
# Step 2: Install cmake and numpy
!apt-get install -y cmake > /dev/null 2>&1
!pip install numpy -q
print("Dependencies ready.")

In [ ]:
# Step 3: Create project directory structure
!mkdir -p /content/pycpp/src
!mkdir -p /content/pycpp/include
!mkdir -p /content/pycpp/build
!mkdir -p /content/pycpp/python/ctypes
!mkdir -p /content/pycpp/python/wrapper
!mkdir -p /content/pycpp/python/wrapper_np
print("Directory structure created.")

In [ ]:
%%writefile /content/pycpp/include/vector_add.h
#ifndef VEC_ADD_H
#define VEC_ADD_H

extern "C" {
    void vectorAdd(int *a, int *b, int *c, int N);
}

#endif

In [ ]:
%%writefile /content/pycpp/src/vector_add.cu
#include <iostream>
#include <cuda_runtime.h>
#include "../include/vector_add.h"

__global__ void vectorAddKernel(int *a, int *b, int *c, int N) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;
    if (idx < N) {
        c[idx] = a[idx] * b[idx];
    }
}

void vectorAdd(int *a, int *b, int *c, int N) {
    int *d_a, *d_b, *d_c;

    cudaMalloc((void**)&d_a, N * sizeof(int));
    cudaMalloc((void**)&d_b, N * sizeof(int));
    cudaMalloc((void**)&d_c, N * sizeof(int));

    cudaMemcpy(d_a, a, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, N * sizeof(int), cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    vectorAddKernel<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, N);

    cudaMemcpy(c, d_c, N * sizeof(int), cudaMemcpyDeviceToHost);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
}

In [ ]:
%%writefile /content/pycpp/CMakeLists.txt
cmake_minimum_required(VERSION 3.10)
project(exposing_functions CUDA)

add_library(vector_add SHARED src/vector_add.cu)
set_target_properties(vector_add PROPERTIES CUDA_SEPARABLE_COMPILATION ON)
target_include_directories(vector_add PUBLIC include)

In [ ]:
# Step 4: Build the CUDA shared library
!cd /content/pycpp && cmake -B build . && cd build && make

---
## Way 1: ctypes

Python's built-in `ctypes` module loads `libvector_add.so` at runtime and calls `vectorAdd` directly.

- **No separate build step** — just load the `.so` and declare the argument types
- Uses NumPy arrays as the data source; passes raw C pointers via `.ctypes.data_as()`

In [ ]:
import ctypes
import numpy as np
import time

# Load the compiled shared library
lib = ctypes.CDLL("/content/pycpp/build/libvector_add.so")

# Declare argument types so ctypes knows how to marshal the call
lib.vectorAdd.argtypes = [
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.POINTER(ctypes.c_int),
    ctypes.c_int
]

# --- Show result with small N ---
N = 10
a = np.array(range(N), dtype=np.int32)
b = np.array(range(N, 2 * N), dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

lib.vectorAdd(
    a.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    b.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    c.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
    N
)

print("=== Way 1: ctypes ===")
print(f"  a         = {a.tolist()}")
print(f"  b         = {b.tolist()}")
print(f"  c = a*b   = {c.tolist()}")

# --- Timing with large N ---
for size in [1_000, 100_000, 1_000_000]:
    a2 = np.array(range(size), dtype=np.int32)
    b2 = np.array(range(size, 2 * size), dtype=np.int32)
    c2 = np.zeros(size, dtype=np.int32)
    t0 = time.time()
    lib.vectorAdd(
        a2.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
        b2.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
        c2.ctypes.data_as(ctypes.POINTER(ctypes.c_int)),
        size
    )
    print(f"  Time (N={size:>9,}): {(time.time() - t0) * 1000:.2f} ms")

---
## Way 2: Python C Extension Wrapper

A custom C file (`vector_add_wrapper.c`) is compiled by `setup.py` into a native Python module.

- Accepts **Python lists** as input
- The C code manually converts lists to C arrays, calls the CUDA function, then returns a new list
- Requires a `setup.py build_ext --inplace` build step

In [ ]:
%%writefile /content/pycpp/python/wrapper/vector_add_wrapper.c
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <dlfcn.h>
#include <stdio.h>
#include <stdlib.h>

typedef void (*vectorAddFunc)(int *, int *, int *, int);

static vectorAddFunc vectorAdd = NULL;

static PyObject* pyVectorAdd(PyObject* self, PyObject* args) {
    PyObject *py_a, *py_b;
    int N;

    if (!PyArg_ParseTuple(args, "OOi", &py_a, &py_b, &N))
        return NULL;

    if (!PyList_Check(py_a) || !PyList_Check(py_b) ||
        PyList_Size(py_a) != N || PyList_Size(py_b) != N) {
        PyErr_SetString(PyExc_ValueError, "Arguments must be lists of same size N.");
        return NULL;
    }

    int *a = (int*)malloc(N * sizeof(int));
    int *b = (int*)malloc(N * sizeof(int));
    int *c = (int*)malloc(N * sizeof(int));

    for (int i = 0; i < N; i++) {
        a[i] = (int)PyLong_AsLong(PyList_GetItem(py_a, i));
        b[i] = (int)PyLong_AsLong(PyList_GetItem(py_b, i));
    }

    vectorAdd(a, b, c, N);

    PyObject* result = PyList_New(N);
    for (int i = 0; i < N; i++) {
        PyList_SetItem(result, i, PyLong_FromLong(c[i]));
    }

    free(a);
    free(b);
    free(c);

    return result;
}

static PyMethodDef VectorAddMethods[] = {
    {"vectorAdd", pyVectorAdd, METH_VARARGS, "Element-wise multiply via CUDA"},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef vectoraddmodule = {
    PyModuleDef_HEAD_INIT,
    "vector_add_wrapper",
    NULL,
    -1,
    VectorAddMethods
};

PyMODINIT_FUNC PyInit_vector_add_wrapper(void) {
    void *handle = dlopen("/content/pycpp/build/libvector_add.so", RTLD_LAZY);
    if (!handle) {
        PyErr_SetString(PyExc_ImportError, "Could not load libvector_add.so");
        return NULL;
    }

    vectorAdd = (vectorAddFunc)dlsym(handle, "vectorAdd");
    if (!vectorAdd) {
        PyErr_SetString(PyExc_ImportError, "Could not find vectorAdd in libvector_add.so");
        return NULL;
    }

    return PyModule_Create(&vectoraddmodule);
}

In [ ]:
%%writefile /content/pycpp/python/wrapper/setup.py
from setuptools import setup, Extension

module = Extension(
    "vector_add_wrapper",
    sources=["vector_add_wrapper.c"],
)

setup(
    name="vector_add_wrapper",
    version="1.0",
    description="Python binding for CUDA vector multiplication",
    ext_modules=[module],
)

In [ ]:
# Build the C extension wrapper
!cd /content/pycpp/python/wrapper && python3 setup.py build_ext --inplace 2>&1 | tail -6

In [ ]:
import sys
import time

sys.path.insert(0, '/content/pycpp/python/wrapper')
import vector_add_wrapper as vaw

# --- Show result with small N ---
N = 10
a = list(range(N))
b = list(range(N, 2 * N))

result = vaw.vectorAdd(a, b, N)

print("=== Way 2: C Extension Wrapper ===")
print(f"  a         = {a}")
print(f"  b         = {b}")
print(f"  c = a*b   = {result}")

# --- Timing with large N ---
for size in [1_000, 100_000, 1_000_000]:
    a2 = list(range(size))
    b2 = list(range(size, 2 * size))
    t0 = time.time()
    vaw.vectorAdd(a2, b2, size)
    print(f"  Time (N={size:>9,}): {(time.time() - t0) * 1000:.2f} ms")

---
## Way 3: NumPy C Extension Wrapper

Same concept as Way 2, but the C extension accepts **NumPy arrays** via `PyArrayObject`.

- Calls `PyArray_DATA()` to get a direct C pointer — **no copy of data needed**
- `c` array is passed in and modified in-place by the CUDA kernel
- Most efficient of the three for large arrays

In [ ]:
%%writefile /content/pycpp/python/wrapper_np/vector_add_np_wrapper.c
#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <numpy/arrayobject.h>
#include <dlfcn.h>
#include <stdio.h>
#include <stdlib.h>

typedef void (*vectorAddFunc)(int *, int *, int *, int);

static vectorAddFunc vectorAdd = NULL;

static PyObject* pyVectorAdd(PyObject* self, PyObject* args) {
    PyArrayObject *a, *b, *c;
    int N;

    if (!PyArg_ParseTuple(args, "OOOi", &a, &b, &c, &N))
        return NULL;

    int *a_ptr = (int*)PyArray_DATA(a);
    int *b_ptr = (int*)PyArray_DATA(b);
    int *c_ptr = (int*)PyArray_DATA(c);

    vectorAdd(a_ptr, b_ptr, c_ptr, N);

    Py_RETURN_NONE;
}

static PyMethodDef VectorAddMethods[] = {
    {"vectorAdd", pyVectorAdd, METH_VARARGS, "Element-wise multiply via CUDA (NumPy)"},
    {NULL, NULL, 0, NULL}
};

static struct PyModuleDef vectoraddmodule = {
    PyModuleDef_HEAD_INIT,
    "vector_add_np_wrapper",
    NULL,
    -1,
    VectorAddMethods
};

PyMODINIT_FUNC PyInit_vector_add_np_wrapper(void) {
    void *handle = dlopen("/content/pycpp/build/libvector_add.so", RTLD_LAZY);
    if (!handle) {
        PyErr_SetString(PyExc_ImportError, "Could not load libvector_add.so");
        return NULL;
    }

    vectorAdd = (vectorAddFunc)dlsym(handle, "vectorAdd");
    if (!vectorAdd) {
        PyErr_SetString(PyExc_ImportError, "Could not find vectorAdd in libvector_add.so");
        return NULL;
    }

    import_array();

    return PyModule_Create(&vectoraddmodule);
}

In [ ]:
%%writefile /content/pycpp/python/wrapper_np/setup.py
from setuptools import setup, Extension
import numpy

module = Extension(
    "vector_add_np_wrapper",
    include_dirs=[numpy.get_include()],
    sources=["vector_add_np_wrapper.c"],
)

setup(
    name="vector_add_np_wrapper",
    version="1.0",
    description="Python binding for CUDA vector multiplication (NumPy)",
    ext_modules=[module],
    include_dirs=[numpy.get_include()]
)

In [ ]:
# Build the NumPy C extension wrapper
!cd /content/pycpp/python/wrapper_np && python3 setup.py build_ext --inplace 2>&1 | tail -6

In [ ]:
import sys
import numpy as np
import time

sys.path.insert(0, '/content/pycpp/python/wrapper_np')
import vector_add_np_wrapper as vaw_np

# --- Show result with small N ---
N = 10
a = np.array(range(N), dtype=np.int32)
b = np.array(range(N, 2 * N), dtype=np.int32)
c = np.zeros(N, dtype=np.int32)

vaw_np.vectorAdd(a, b, c, N)

print("=== Way 3: NumPy C Extension Wrapper ===")
print(f"  a         = {a.tolist()}")
print(f"  b         = {b.tolist()}")
print(f"  c = a*b   = {c.tolist()}")

# --- Timing with large N ---
for size in [1_000, 100_000, 1_000_000]:
    a2 = np.array(range(size), dtype=np.int32)
    b2 = np.array(range(size, 2 * size), dtype=np.int32)
    c2 = np.zeros(size, dtype=np.int32)
    t0 = time.time()
    vaw_np.vectorAdd(a2, b2, c2, size)
    print(f"  Time (N={size:>9,}): {(time.time() - t0) * 1000:.2f} ms")

---
## Summary

All three methods successfully called the CUDA kernel `vectorAddKernel` from Python.

| Method | Input type | Data copy to C? | Build step? |
|--------|------------|-----------------|-------------|
| ctypes | NumPy array | No (uses raw pointer) | No |
| C Extension Wrapper | Python list | Yes (list → malloc array) | Yes (`setup.py`) |
| NumPy C Extension Wrapper | NumPy array | No (uses `PyArray_DATA`) | Yes (`setup.py`) |

The CUDA kernel runs `c[i] = a[i] * b[i]` for all `i` simultaneously on the GPU.